In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

processed_data_path = (
    project_root
    / "data"
    / "processed"
    / "heart_disease_clean.csv"
)

if not processed_data_path.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {processed_data_path}"
    )

df = pd.read_csv(processed_data_path)

target_column = "heart_disease"

if target_column not in df.columns:
    raise KeyError(
        f"Target column '{target_column}' was not found."
    )

if df[target_column].isna().any():
    raise ValueError("The target column contains missing values.")

X = df.drop(columns=[target_column])
y = df[target_column].astype("int64")

print("Full dataset shape:", df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Feature count:", X.shape[1])
print("Target classes:", sorted(y.unique().tolist()))
print("Missing feature values:", int(X.isna().sum().sum()))

Full dataset shape: (4238, 16)
Feature matrix shape: (4238, 15)
Target shape: (4238,)
Feature count: 15
Target classes: [0, 1]
Missing feature values: 645


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

train_distribution = (
    y_train.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

test_distribution = (
    y_test.value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

distribution_check = pd.DataFrame({
    "Class": [0, 1],
    "Training Percentage": [
        train_distribution.get(0, 0),
        train_distribution.get(1, 0)
    ],
    "Testing Percentage": [
        test_distribution.get(0, 0),
        test_distribution.get(1, 0)
    ]
})

distribution_check

X_train shape: (3390, 15)
X_test shape: (848, 15)
y_train shape: (3390,)
y_test shape: (848,)


,Class,Training Percentage,Testing Percentage
0,0,84.81,84.79
1,1,15.19,15.21


In [3]:
continuous_features = [
    "age",
    "cigarettes_per_day",
    "total_cholesterol",
    "systolic_bp",
    "diastolic_bp",
    "bmi",
    "heart_rate",
    "glucose"
]

binary_features = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

categorical_features = [
    "gender",
    "education"
]

all_defined_features = (
    continuous_features
    + binary_features
    + categorical_features
)

missing_features = sorted(
    set(X_train.columns) - set(all_defined_features)
)

unexpected_features = sorted(
    set(all_defined_features) - set(X_train.columns)
)

duplicate_features = [
    feature
    for feature in set(all_defined_features)
    if all_defined_features.count(feature) > 1
]

if missing_features:
    raise ValueError(
        f"Features not assigned to a group: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"Defined features not found in dataset: {unexpected_features}"
    )

if duplicate_features:
    raise ValueError(
        f"Features assigned more than once: {duplicate_features}"
    )

feature_group_summary = pd.DataFrame({
    "Feature Group": [
        "Continuous Numerical",
        "Binary Indicator",
        "Categorical"
    ],
    "Feature Count": [
        len(continuous_features),
        len(binary_features),
        len(categorical_features)
    ],
    "Features": [
        continuous_features,
        binary_features,
        categorical_features
    ]
})

display(feature_group_summary)

print("Total features defined:", len(all_defined_features))
print("Total dataset features:", X_train.shape[1])

,Feature Group,Feature Count,Features
0,Continuous Numerical,8,"[age, cigarettes_per_day, total_cholesterol, s..."
1,Binary Indicator,5,"[current_smoker, bp_meds, prevalent_stroke, pr..."
2,Categorical,2,"[gender, education]"


Total features defined: 15
Total dataset features: 15


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_pipeline,
            continuous_features
        ),
        (
            "binary",
            binary_pipeline,
            binary_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

print(preprocessor)

ColumnTransformer(transformers=[('continuous',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'cigarettes_per_day',
                                  'total_cholesterol', 'systolic_bp',
                                  'diastolic_bp', 'bmi', 'heart_rate',
                                  'glucose']),
                                ('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent'))]),
                                 ['current_smoker', 'bp_meds',
                                  'prevalent_stroke', 'prevalent_hypertension',
                                  'diabetes']),
                                ('categorical',
                                 Pipel